# Federated Learning: Privacy-Preserving Machine Learning

## What is Federated Learning?
Federated Learning (FL) trains a model across many decentralized devices or servers holding local data, without centralizing raw data. The data never leaves the device.

## Why Federated Learning?
- **Privacy**: Raw data stays on device (GDPR, HIPAA compliance)
- **Data sovereignty**: Organizations keep data in-house
- **Scale**: Learn from millions of devices/hospitals
- **Communication efficiency**: Only share model updates, not data

## Federated Learning Architecture
```
       ┌────────────────────────────┐
       │    Central Server          │
       │  (aggregates updates)      │
       └─────────┬──────────────────┘
                 │ Global Model w_t
        ┌────────┼────────────────┐
        ▼        ▼                ▼
   Client 1  Client 2  ...  Client K
   Local D1  Local D2  ...  Local DK
   Trains    Trains    ...  Trains
   sends Δw1 sends Δw2     sends ΔwK
```

## FedAvg Algorithm

The foundational algorithm by McMahan et al. (2017):

**Server round $t$:**
$$w_{t+1} = \sum_{k=1}^{K} \frac{n_k}{n} w_{t+1}^k$$

**Client $k$ update (E local steps):**
$$w_{t+1}^k = w_t - \eta \nabla F_k(w_t)$$

Where:
- $n_k$ = number of samples on client $k$
- $n = \sum_k n_k$ = total samples
- $F_k$ = local objective on client $k$
- $E$ = number of local epochs

In [1]:
import numpy as np
import copy
from typing import List, Tuple

# FedAvg from scratch
class FederatedServer:
    def __init__(self, global_weights):
        self.global_weights = copy.deepcopy(global_weights)
    
    def aggregate(self, client_updates: List[Tuple[np.ndarray, int]]):
        """FedAvg aggregation: weighted average by dataset size."""
        total_samples = sum(n for _, n in client_updates)
        new_weights = [np.zeros_like(w) for w in self.global_weights]
        
        for client_weights, n_samples in client_updates:
            weight = n_samples / total_samples
            for i, w in enumerate(client_weights):
                new_weights[i] += weight * w
        
        self.global_weights = new_weights
        return self.global_weights

class FederatedClient:
    def __init__(self, client_id: int, X, y, learning_rate=0.01, local_epochs=5):
        self.client_id = client_id
        self.X = X
        self.y = y
        self.lr = learning_rate
        self.local_epochs = local_epochs
        self.weights = None
    
    def set_weights(self, global_weights):
        self.weights = copy.deepcopy(global_weights)
    
    def train_local(self):
        """Simple logistic regression with SGD."""
        W, b = self.weights
        for _ in range(self.local_epochs):
            for xi, yi in zip(self.X, self.y):
                z = np.dot(W, xi) + b
                pred = 1 / (1 + np.exp(-z))
                error = pred - yi
                W -= self.lr * error * xi
                b -= self.lr * error
        self.weights = [W, b]
        return self.weights, len(self.X)

print('Federated classes defined')

Federated classes defined


In [2]:
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Load and preprocess data
data = load_breast_cancer()
X, y = data.data, data.target.astype(float)
scaler = StandardScaler()
X = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Simulate non-IID data distribution across clients
n_clients = 5
client_data = []
indices = np.random.permutation(len(X_train))
split_indices = np.array_split(indices, n_clients)

for client_indices in split_indices:
    client_data.append((X_train[client_indices], y_train[client_indices]))

print(f'{n_clients} clients with sizes: {[len(d[0]) for d in client_data]}')

# Initialize
n_features = X.shape[1]
initial_weights = [np.zeros(n_features), np.zeros(1)]

server = FederatedServer(initial_weights)
clients = [FederatedClient(i, *client_data[i]) for i in range(n_clients)]

# Federated training rounds
def federated_predict(weights, X):
    W, b = weights
    z = X @ W + b
    return (1 / (1 + np.exp(-z)) > 0.5).astype(int)

for round_num in range(10):
    # Server distributes global weights
    client_updates = []
    for client in clients:
        client.set_weights(server.global_weights)
        updated_weights, n_samples = client.train_local()
        client_updates.append((updated_weights, n_samples))
    
    # Server aggregates
    server.aggregate(client_updates)
    
    # Evaluate global model
    y_pred = federated_predict(server.global_weights, X_test)
    acc = accuracy_score(y_test, y_pred)
    if (round_num + 1) % 2 == 0:
        print(f'Round {round_num+1}: Test Accuracy = {acc:.4f}')

5 clients with sizes: [91, 91, 91, 91, 91]


Round 2: Test Accuracy = 0.9825


Round 4: Test Accuracy = 0.9912


Round 6: Test Accuracy = 0.9912


Round 8: Test Accuracy = 0.9825


Round 10: Test Accuracy = 0.9825


## Challenges in Federated Learning

### Non-IID Data
Real clients have heterogeneous data distributions. Solutions:
- **FedProx**: Add proximal term to prevent local models from drifting too far
  $$\min_w F_k(w) + \frac{\mu}{2}||w - w^t||^2$$
- **FedNova**: Normalize updates by number of local steps
- **SCAFFOLD**: Correct for client drift with control variates

### Communication Efficiency
- Gradient compression: quantization, sparsification
- Model pruning before transmission
- **FedPAQ**: Periodic averaging + quantization

### System Heterogeneity
- Different devices have different compute/memory
- Asynchronous FL: **FedAsync**, **FedBuff**

## Differential Privacy in FL

Differential Privacy (DP) provides mathematical privacy guarantees:

$$\Pr[\mathcal{M}(D) \in S] \leq e^\epsilon \cdot \Pr[\mathcal{M}(D') \in S] + \delta$$

Where $D$ and $D'$ differ in one record. Smaller $\epsilon$ = stronger privacy.

### Gaussian Mechanism
Add calibrated noise to gradients:
$$\tilde{g} = g + \mathcal{N}(0, \sigma^2 C^2 I)$$

Where $C$ is the gradient clipping threshold and $\sigma$ is the noise multiplier.

```python
# DP with PyTorch Opacus
# pip install opacus
from opacus import PrivacyEngine

model = MyModel()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
dataloader = DataLoader(dataset, batch_size=64)

privacy_engine = PrivacyEngine()
model, optimizer, dataloader = privacy_engine.make_private(
    module=model,
    optimizer=optimizer,
    data_loader=dataloader,
    noise_multiplier=1.1,   # σ controls privacy-utility tradeoff
    max_grad_norm=1.0,      # C gradient clipping
)

for epoch in range(20):
    for batch in dataloader:
        loss = compute_loss(model, batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()  # Opacus handles DP noise addition

epsilon = privacy_engine.get_epsilon(delta=1e-5)
print(f'(ε={epsilon:.2f}, δ=1e-5)-DP guarantee')
```

## Flower Framework

```python
# pip install flwr[simulation]
import flwr as fl
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# Define Flower client
class FlowerClient(fl.client.NumPyClient):
    def __init__(self, net, trainloader, testloader):
        self.net = net
        self.trainloader = trainloader
        self.testloader = testloader
    
    def get_parameters(self, config):
        return [val.cpu().numpy() for _, val in self.net.state_dict().items()]
    
    def set_parameters(self, parameters):
        params_dict = zip(self.net.state_dict().keys(), parameters)
        state_dict = {k: torch.tensor(v) for k, v in params_dict}
        self.net.load_state_dict(state_dict, strict=True)
    
    def fit(self, parameters, config):
        self.set_parameters(parameters)
        train(self.net, self.trainloader, epochs=config.get('epochs', 1))
        return self.get_parameters(config), len(self.trainloader.dataset), {}
    
    def evaluate(self, parameters, config):
        self.set_parameters(parameters)
        loss, accuracy = test(self.net, self.testloader)
        return loss, len(self.testloader.dataset), {'accuracy': accuracy}

# Define strategy (FedAvg)
strategy = fl.server.strategy.FedAvg(
    fraction_fit=0.1,           # 10% clients per round
    fraction_evaluate=0.05,
    min_fit_clients=5,
    min_evaluate_clients=3,
    min_available_clients=10,
)

# Simulate federation
fl.simulation.start_simulation(
    client_fn=lambda cid: FlowerClient(...),
    num_clients=100,
    config=fl.server.ServerConfig(num_rounds=10),
    strategy=strategy,
)
```

## Additional Learning Resources

### Papers
- [Communication-Efficient Learning of Deep Networks from Decentralized Data (FedAvg)](https://arxiv.org/abs/1602.05629)
- [Advances and Open Problems in Federated Learning](https://arxiv.org/abs/1912.04977) 60+ authors, comprehensive survey
- [Deep Learning with Differential Privacy](https://arxiv.org/abs/1607.00133)
- [FedProx](https://arxiv.org/abs/1812.06127)

### Tools
- [Flower Docs](https://flower.ai/docs/) FL framework
- [PySyft Docs](https://pysyft.readthedocs.io/) Privacy-preserving ML
- [TensorFlow Federated](https://www.tensorflow.org/federated)
- [Opacus Docs](https://opacus.ai/) DP training with PyTorch

### Courses
- [Federated Learning PriML Workshop](https://priml-workshop.github.io/)
- [Privacy in ML Stanford CS329D](https://cs329d.stanford.edu/)